In [1]:
from rdkit import Chem

In [2]:
import pandas as pd 

In [3]:
import pandas as pd
df = pd.read_csv("Brain.csv")
df.head()

,smiles
0,CCOc1ccc2nc3cc(N)ccc3c(N)c2c1
1,c1ccc(CN(CC2=NCCN2)c2ccccc2)cc1
2,CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1
3,c1ccc(C2CC2)c(OCC2=NCCN2)c1
4,CC(C)NCC(O)c1ccccc1Cl


In [4]:
print(df["smiles"].isnull().sum())

0


In [5]:
from rdkit import Chem
import pandas as pd
df = pd.read_csv("Brain.csv")
df.columns = df.columns.str.strip()
def canonicalize_smiles(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        return Chem.MolToSmiles(mol, canonical=True)
    else:
        return None
df["canonical_smiles"] = df["smiles"].apply(canonicalize_smiles)
df.to_csv("canonical_smiles_outputBrain.csv", index=False)
print(df.head())

                               smiles                    canonical_smiles
0       CCOc1ccc2nc3cc(N)ccc3c(N)c2c1       CCOc1ccc2nc3cc(N)ccc3c(N)c2c1
1     c1ccc(CN(CC2=NCCN2)c2ccccc2)cc1     c1ccc(CN(CC2=NCCN2)c2ccccc2)cc1
2  CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1  CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1
3         c1ccc(C2CC2)c(OCC2=NCCN2)c1         c1ccc(C2CC2)c(OCC2=NCCN2)c1
4               CC(C)NCC(O)c1ccccc1Cl               CC(C)NCC(O)c1ccccc1Cl


In [6]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
import pandas as pd
import numpy as np
def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return pd.Series({
            "MolWt": np.nan,
            "LogP": np.nan,
            "H_Donors": np.nan,
            "H_Acceptors": np.nan,
            "TPSA": np.nan,
            "NumRotatableBonds": np.nan,

            "NumAtoms": np.nan,
            "HeavyAtomCount": np.nan,
            "NumHeteroatoms": np.nan,
            "NumCarbons": np.nan,
            "NumNitrogens": np.nan,
            "NumOxygens": np.nan,
            "NumHalogens": np.nan,

            "NumBonds": np.nan,
            "NumDoubleBonds": np.nan,
            "NumTripleBonds": np.nan,

            "NumRings": np.nan,
            "NumAromaticRings": np.nan,
            "NumAliphaticRings": np.nan,
            "NumSaturatedRings": np.nan,

            "NumAromaticAtoms": np.nan,
            "NumAliphaticAtoms": np.nan,

            "FractionCSP3": np.nan
        })

    MolWt = Descriptors.MolWt(mol)

    LogP = Descriptors.MolLogP(mol)

    H_Donors = Descriptors.NumHDonors(mol)

    H_Acceptors = Descriptors.NumHAcceptors(mol)

    TPSA = Descriptors.TPSA(mol)

    NumRotatableBonds = Descriptors.NumRotatableBonds(mol)

    NumAtoms = mol.GetNumAtoms()

    HeavyAtomCount = mol.GetNumHeavyAtoms()


    # Count heteroatoms
    NumHeteroatoms = sum(
        1 for atom in mol.GetAtoms()
        if atom.GetAtomicNum() not in [1, 6]
    )


    # Carbon atoms
    NumCarbons = sum(
        1 for atom in mol.GetAtoms()
        if atom.GetAtomicNum() == 6
    )


    # Nitrogen atoms
    NumNitrogens = sum(
        1 for atom in mol.GetAtoms()
        if atom.GetAtomicNum() == 7
    )


    # Oxygen atoms
    NumOxygens = sum(
        1 for atom in mol.GetAtoms()
        if atom.GetAtomicNum() == 8
    )


    # Halogen atoms: F, Cl, Br, I
    NumHalogens = sum(
        1 for atom in mol.GetAtoms()
        if atom.GetAtomicNum() in [9, 17, 35, 53]
    )


    # ========================================================
    # 3. BOND COUNTS
    # ========================================================

    NumBonds = mol.GetNumBonds()


    NumDoubleBonds = sum(
        1 for bond in mol.GetBonds()
        if bond.GetBondType() == Chem.BondType.DOUBLE
    )


    NumTripleBonds = sum(
        1 for bond in mol.GetBonds()
        if bond.GetBondType() == Chem.BondType.TRIPLE
    )

    NumRings = rdMolDescriptors.CalcNumRings(mol)

    NumAromaticRings = rdMolDescriptors.CalcNumAromaticRings(mol)

    NumAliphaticRings = rdMolDescriptors.CalcNumAliphaticRings(mol)

    NumSaturatedRings = rdMolDescriptors.CalcNumSaturatedRings(mol)

    NumAromaticAtoms = sum(
        1 for atom in mol.GetAtoms()
        if atom.GetIsAromatic()
    )


    NumAliphaticAtoms = sum(
        1 for atom in mol.GetAtoms()
        if not atom.GetIsAromatic()
    )


    FractionCSP3 = rdMolDescriptors.CalcFractionCSP3(mol)



    return pd.Series({

        # Lipinski
        "MolWt": MolWt,
        "LogP": LogP,
        "H_Donors": H_Donors,
        "H_Acceptors": H_Acceptors,
        "TPSA": TPSA,
        "NumRotatableBonds": NumRotatableBonds,

        # Atom counts
        "NumAtoms": NumAtoms,
        "HeavyAtomCount": HeavyAtomCount,
        "NumHeteroatoms": NumHeteroatoms,
        "NumCarbons": NumCarbons,
        "NumNitrogens": NumNitrogens,
        "NumOxygens": NumOxygens,
        "NumHalogens": NumHalogens,

        # Bond counts
        "NumBonds": NumBonds,
        "NumDoubleBonds": NumDoubleBonds,
        "NumTripleBonds": NumTripleBonds,

        # Ring counts
        "NumRings": NumRings,
        "NumAromaticRings": NumAromaticRings,
        "NumAliphaticRings": NumAliphaticRings,
        "NumSaturatedRings": NumSaturatedRings,

        # Aromaticity
        "NumAromaticAtoms": NumAromaticAtoms,
        "NumAliphaticAtoms": NumAliphaticAtoms,

        # Carbon hybridization
        "FractionCSP3": FractionCSP3
    })

descriptor_columns = [
    "MolWt",
    "LogP",
    "H_Donors",
    "H_Acceptors",
    "TPSA",
    "NumRotatableBonds",

    "NumAtoms",
    "HeavyAtomCount",
    "NumHeteroatoms",
    "NumCarbons",
    "NumNitrogens",
    "NumOxygens",
    "NumHalogens",

    "NumBonds",
    "NumDoubleBonds",
    "NumTripleBonds",

    "NumRings",
    "NumAromaticRings",
    "NumAliphaticRings",
    "NumSaturatedRings",

    "NumAromaticAtoms",
    "NumAliphaticAtoms",

    "FractionCSP3"
]


df[descriptor_columns] = df["canonical_smiles"].apply(
    calculate_descriptors
)

df.to_csv(
    "Brain_descriptors.csv",
    index=False
)
print("Descriptor calculation completed successfully!")

print("\nNumber of descriptors generated:",
      len(descriptor_columns))

print("\nDescriptor columns:")
print(descriptor_columns)

print("\nFirst 5 rows:")
display(df[descriptor_columns].head())

print("\nFinal dataset shape:")
print(df.shape)

Descriptor calculation completed successfully!

Number of descriptors generated: 23

Descriptor columns:
['MolWt', 'LogP', 'H_Donors', 'H_Acceptors', 'TPSA', 'NumRotatableBonds', 'NumAtoms', 'HeavyAtomCount', 'NumHeteroatoms', 'NumCarbons', 'NumNitrogens', 'NumOxygens', 'NumHalogens', 'NumBonds', 'NumDoubleBonds', 'NumTripleBonds', 'NumRings', 'NumAromaticRings', 'NumAliphaticRings', 'NumSaturatedRings', 'NumAromaticAtoms', 'NumAliphaticAtoms', 'FractionCSP3']

First 5 rows:


,MolWt,LogP,H_Donors,H_Acceptors,TPSA,NumRotatableBonds,NumAtoms,HeavyAtomCount,NumHeteroatoms,NumCarbons,...,NumBonds,NumDoubleBonds,NumTripleBonds,NumRings,NumAromaticRings,NumAliphaticRings,NumSaturatedRings,NumAromaticAtoms,NumAliphaticAtoms,FractionCSP3
0,253.305,2.9511,2.0,4.0,74.16,2.0,19.0,19.0,4.0,15.0,...,21.0,0.0,0.0,3.0,3.0,0.0,0.0,14.0,5.0,0.133333
1,265.360,2.6949,1.0,3.0,27.63,5.0,20.0,20.0,3.0,17.0,...,22.0,1.0,0.0,3.0,2.0,1.0,0.0,12.0,8.0,0.235294
2,291.347,2.3731,2.0,5.0,71.70,7.0,21.0,21.0,5.0,16.0,...,22.0,1.0,0.0,2.0,2.0,0.0,0.0,9.0,12.0,0.437500
3,216.284,1.9445,1.0,3.0,33.62,4.0,16.0,16.0,3.0,13.0,...,18.0,1.0,0.0,3.0,1.0,2.0,1.0,6.0,10.0,0.461538
4,213.708,2.3714,2.0,2.0,32.26,4.0,14.0,14.0,3.0,11.0,...,14.0,0.0,0.0,1.0,1.0,0.0,0.0,6.0,8.0,0.454545



Final dataset shape:
(77, 25)


# fingerprints

In [7]:
df_smiles = pd.read_csv("canonical_smiles_outputBrain.csv")
print(df_smiles.columns)

Index(['smiles', 'canonical_smiles'], dtype='str')


In [8]:
df_desc = pd.read_csv("Brain_descriptors.csv")
df_smiles = pd.read_csv("canonical_smiles_outputBrain.csv")

df_desc["canonical_smiles"] = df_smiles["canonical_smiles"]

df_desc.to_csv("final_with_descriptorsBrain_updated.csv", index=False)

print("✅ canonical_smiles added successfully")
print(df_desc.head())

✅ canonical_smiles added successfully
                               smiles                    canonical_smiles  \
0       CCOc1ccc2nc3cc(N)ccc3c(N)c2c1       CCOc1ccc2nc3cc(N)ccc3c(N)c2c1   
1     c1ccc(CN(CC2=NCCN2)c2ccccc2)cc1     c1ccc(CN(CC2=NCCN2)c2ccccc2)cc1   
2  CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1  CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1   
3         c1ccc(C2CC2)c(OCC2=NCCN2)c1         c1ccc(C2CC2)c(OCC2=NCCN2)c1   
4               CC(C)NCC(O)c1ccccc1Cl               CC(C)NCC(O)c1ccccc1Cl   

     MolWt    LogP  H_Donors  H_Acceptors   TPSA  NumRotatableBonds  NumAtoms  \
0  253.305  2.9511       2.0          4.0  74.16                2.0      19.0   
1  265.360  2.6949       1.0          3.0  27.63                5.0      20.0   
2  291.347  2.3731       2.0          5.0  71.70                7.0      21.0   
3  216.284  1.9445       1.0          3.0  33.62                4.0      16.0   
4  213.708  2.3714       2.0          2.0  32.26                4.0      14.0   

   HeavyAtom

In [9]:
from rdkit import Chem
from rdkit.Chem import MACCSkeys
import pandas as pd
import numpy as np
df = pd.read_csv("final_with_descriptorsBrain_updated.csv")

# Function
def maccs_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return np.array(MACCSkeys.GenMACCSKeys(mol))
    else:
        return np.zeros(167)

# Apply
maccs = df["canonical_smiles"].apply(maccs_fp)

# Convert to DataFrame
maccs_df = pd.DataFrame(maccs.tolist())
maccs_df.columns = [f"maccs_{i}" for i in range(maccs_df.shape[1])]

# Add to main dataset
df = pd.concat([df, maccs_df], axis=1)

# Save
df.to_csv("step1_maccs_addedBrain.csv", index=False)

print("✅ MACCS done")
print(df.shape)

✅ MACCS done
(77, 192)


In [10]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import pandas as pd
import numpy as np

# Load your dataset (must contain canonical_smiles)
df = pd.read_csv("final_with_descriptorsBrain_updated.csv")

# Initialize Morgan generator (NEW method)
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# Function to generate fingerprint
def morgan_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return np.array(morgan_gen.GetFingerprint(mol))
    else:
        return np.zeros(2048)

# Apply on canonical_smiles column
morgan = df["canonical_smiles"].apply(morgan_fp)

# Convert to DataFrame
morgan_df = pd.DataFrame(morgan.tolist())

# Rename columns
morgan_df.columns = [f"morgan_{i}" for i in range(morgan_df.shape[1])]

# Combine with original dataset
df_final = pd.concat([df, morgan_df], axis=1)

# Save output
df_final.to_csv("morgan_addedBrain.csv", index=False)

print("✅ Morgan fingerprint added successfully")
print("Shape:", df_final.shape)
print(df_final.head())

✅ Morgan fingerprint added successfully
Shape: (77, 2073)
                               smiles                    canonical_smiles  \
0       CCOc1ccc2nc3cc(N)ccc3c(N)c2c1       CCOc1ccc2nc3cc(N)ccc3c(N)c2c1   
1     c1ccc(CN(CC2=NCCN2)c2ccccc2)cc1     c1ccc(CN(CC2=NCCN2)c2ccccc2)cc1   
2  CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1  CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1   
3         c1ccc(C2CC2)c(OCC2=NCCN2)c1         c1ccc(C2CC2)c(OCC2=NCCN2)c1   
4               CC(C)NCC(O)c1ccccc1Cl               CC(C)NCC(O)c1ccccc1Cl   

     MolWt    LogP  H_Donors  H_Acceptors   TPSA  NumRotatableBonds  NumAtoms  \
0  253.305  2.9511       2.0          4.0  74.16                2.0      19.0   
1  265.360  2.6949       1.0          3.0  27.63                5.0      20.0   
2  291.347  2.3731       2.0          5.0  71.70                7.0      21.0   
3  216.284  1.9445       1.0          3.0  33.62                4.0      16.0   
4  213.708  2.3714       2.0          2.0  32.26                4.0      1

In [20]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import pandas as pd
import numpy as np

# Load previous dataset (use latest file you created)
df = pd.read_csv("morgan_addedBrain.csv")   # or your latest file

# Initialize AtomPair generator
ap_gen = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=1024)

# Function
def atompair_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return np.array(ap_gen.GetFingerprint(mol))
    else:
        return np.zeros(1024)

# Apply
ap = df["canonical_smiles"].apply(atompair_fp)

# Convert to DataFrame
ap_df = pd.DataFrame(ap.tolist())

# Rename columns
ap_df.columns = [f"atompair_{i}" for i in range(ap_df.shape[1])]
# Combine with dataset
df_final = pd.concat([df, ap_df], axis=1)

# Save
df_final.to_csv("atompair_addedBrain.csv", index=False)
print("✅ AtomPair fingerprint added")
print("Shape:", df_final.shape)

✅ AtomPair fingerprint added
Shape: (77, 3097)


In [21]:
print("External selected feature shape:", X_external_selected.shape)

External selected feature shape: (77, 1844)


In [22]:
# Find the trained Random Forest model
[name for name, obj in globals().items()
 if "RandomForest" in str(type(obj))]

[]

In [23]:
import os

files = os.listdir()

[f for f in files if f.endswith((".pkl", ".joblib"))]

['Androgene_feature_columns.pkl',
 'Androgen_feature_columns.pkl',
 'Androgen_feature_selector.pkl',
 'Androgen_rf_model.pkl',
 'aromatase_feature_columns.pkl',
 'aromatase_feature_selector.pkl',
 'aromatase_rf_model.pkl',
 'bioavailability_DT_model.pkl',
 'bioavailability_feature_columns.pkl',
 'bioavailability_rfecv.pkl',
 'DecisionTree_model.pkl',
 'DecisionTree_newone.pkl',
 'feature_columns.pkl',
 'feature_selector.pkl',
 'finalMTOR_tuned_RF.pkl',
 'final_tuned_RF.pkl',
 'final_tuned_RFpro.pkl',
 'newdrug_feature_columns.pkl',
 'newdrug_feature_selector.pkl',
 'newdrug_rf_model.pkl',
 'rfecv.pkl',
 'rf_model.pkl',
 'sphk1_feature_columns.pkl',
 'sphk1_feature_selector.pkl',
 'sphk1_rf_model.pkl']

In [25]:
import os

[f for f in os.listdir() if "MTOR" in f.upper() or "MTOR" in f.lower()]

['atompair_addedMTOR.csv',
 'canonical_smiles_outputMTOR.csv',
 'external_compound_infoMTOR.csv',
 'finalMTOR_tuned_RF.pkl',
 'final_full_datasetMTOR.csv',
 'final_with_descriptorsMTOR_updated.csv',
 'morgan_addedMTOR.csv',
 'MTOR.csv',
 'MTOR.ipynb',
 'MTOR2.ipynb',
 'MTOR_descriptors.csv',
 'MTOR_Top20_hits.csv',
 'rfecv_selected_featuresMTOR.csv',
 'step1_maccs_addedMTOR.csv']

In [30]:
[f for f in os.listdir() if f.endswith(".pkl") or f.endswith(".joblib")]

['Androgene_feature_columns.pkl',
 'Androgen_feature_columns.pkl',
 'Androgen_feature_selector.pkl',
 'Androgen_rf_model.pkl',
 'aromatase_feature_columns.pkl',
 'aromatase_feature_selector.pkl',
 'aromatase_rf_model.pkl',
 'bioavailability_DT_model.pkl',
 'bioavailability_feature_columns.pkl',
 'bioavailability_rfecv.pkl',
 'DecisionTree_model.pkl',
 'DecisionTree_newone.pkl',
 'feature_columns.pkl',
 'feature_selector.pkl',
 'finalMTOR_tuned_RF.pkl',
 'final_tuned_RF.pkl',
 'final_tuned_RFpro.pkl',
 'newdrug_feature_columns.pkl',
 'newdrug_feature_selector.pkl',
 'newdrug_rf_model.pkl',
 'rfecv.pkl',
 'rf_model.pkl',
 'sphk1_feature_columns.pkl',
 'sphk1_feature_selector.pkl',
 'sphk1_rf_model.pkl']

In [31]:
%pip install imbalanced-learn


   ---------------------------------------- 0/2 [sklearn-compat]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]

Note: you may need to restart the kernel to use updated packages.


In [32]:
import joblib

mtor_rf = joblib.load("finalMTOR_tuned_RF.pkl")

print("✅ MTOR Random Forest loaded")
print("Model type:", type(mtor_rf))

C:\Users\sahan\anaconda3\envs\rdkit-env\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator NearestNeighbors from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\sahan\anaconda3\envs\rdkit-env\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


✅ MTOR Random Forest loaded
Model type: <class 'imblearn.pipeline.Pipeline'>


C:\Users\sahan\anaconda3\envs\rdkit-env\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [33]:
print("Model input features:", mtor_rf.n_features_in_)
print("External matrix:", X_external_selected.shape)

Model input features: 1844
External matrix: (77, 1844)


In [34]:
# ============================================
# MTOR: Predict and rank external compounds
# ============================================

# Predict probability of the positive/active class
hit_scores = mtor_rf.predict_proba(X_external_selected)[:, 1]

# Create results table
top20_results = external_df.copy()

top20_results["Hit_Score"] = hit_scores

# Rank highest score first
top20_results = top20_results.sort_values(
    by="Hit_Score",
    ascending=False
).reset_index(drop=True)

# Add rank
top20_results.insert(
    0,
    "Rank",
    range(1, len(top20_results) + 1)
)

# Select Top 20
top20_hits = top20_results.head(20).copy()

# Display
display(top20_hits)

NameError: name 'external_df' is not defined

In [35]:
# Predict the 77 external compounds
hit_scores = mtor_rf.predict_proba(X_external_selected)[:, 1]

# Create results using the selected-feature matrix
top20_results = X_external_selected.copy()

# Add ML hit score
top20_results["Hit_Score"] = hit_scores

# Rank highest score first
top20_results = top20_results.sort_values(
    by="Hit_Score",
    ascending=False
).reset_index(drop=True)

# Add rank
top20_results.insert(0, "Rank", range(1, len(top20_results) + 1))

# Get Top 20
top20_hits = top20_results.head(20)

display(top20_hits[["Rank", "Hit_Score"]])

,Rank,Hit_Score
0,1,0.737756
1,2,0.663906
2,3,0.655661
3,4,0.654221
4,5,0.654221
5,6,0.646918
6,7,0.640305
7,8,0.640305
8,9,0.634615
9,10,0.634615


In [36]:
# Reload the complete external dataset
external_df = pd.read_csv("atompair_addedBrain.csv")

# Confirm dimensions
print("Complete external data:", external_df.shape)
print("Selected feature matrix:", X_external_selected.shape)

# Predict all 77 compounds
predicted_class = mtor_rf.predict(X_external_selected)

# Probability of positive/active class
probability = mtor_rf.predict_proba(X_external_selected)[:, 1]

# Create complete results
all_results = external_df.copy()

all_results["Predicted_Class"] = predicted_class
all_results["Probability"] = probability

# Rank all compounds by probability
all_results = all_results.sort_values(
    by="Probability",
    ascending=False
).reset_index(drop=True)

all_results.insert(0, "Rank", range(1, len(all_results) + 1))

print("Total compounds:", len(all_results))

display(
    all_results[
        ["Rank", "canonical_smiles", "Predicted_Class", "Probability"]
    ]
)

Complete external data: (77, 3097)
Selected feature matrix: (77, 1844)
Total compounds: 77


,Rank,canonical_smiles,Predicted_Class,Probability
0,1,C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12,1,0.737756
1,2,COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)...,1,0.663906
2,3,C[C@H]1O[C@@]2(CS1)CN1CCC2CC1,1,0.655661
3,4,COc1cc(NC(C)CCCN)c2ncccc2c1,1,0.654221
4,5,COc1cc(NC(C)CCCN)c2ncccc2c1,1,0.654221
...,...,...,...,...
72,73,NCCc1ccc(O)cc1,0,0.295056
73,74,NCCc1ccc(O)cc1,0,0.295056
74,75,C[C@H](N)C(=O)c1ccccc1,0,0.226177
75,76,CC(=O)C(C)NCCc1ccccc1,0,0.189488


In [37]:
# Predict all 77 external compounds
predicted_class = mtor_rf.predict(X_external_selected)
probability = mtor_rf.predict_proba(X_external_selected)[:, 1]

# Create results with SMILES
all_results = pd.DataFrame({
    "SMILES": external_df["canonical_smiles"],
    "Predicted_Class": predicted_class,
    "Probability": probability
})

# Rank by probability
all_results = all_results.sort_values(
    "Probability",
    ascending=False
).reset_index(drop=True)

# Add rank
all_results.insert(0, "Rank", range(1, len(all_results) + 1))

# Display ALL 77
display(all_results)

,Rank,SMILES,Predicted_Class,Probability
0,1,C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12,1,0.737756
1,2,COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)...,1,0.663906
2,3,C[C@H]1O[C@@]2(CS1)CN1CCC2CC1,1,0.655661
3,4,COc1cc(NC(C)CCCN)c2ncccc2c1,1,0.654221
4,5,COc1cc(NC(C)CCCN)c2ncccc2c1,1,0.654221
...,...,...,...,...
72,73,NCCc1ccc(O)cc1,0,0.295056
73,74,NCCc1ccc(O)cc1,0,0.295056
74,75,C[C@H](N)C(=O)c1ccccc1,0,0.226177
75,76,CC(=O)C(C)NCCc1ccccc1,0,0.189488


In [38]:
# ============================================
# TOP 20 MTOR HITS - KEEP ALL ORIGINAL COLUMNS
# ============================================

# Predict all 77 compounds using ONLY the 1844 selected features
predictions = mtor_rf.predict(X_external_selected)
probabilities = mtor_rf.predict_proba(X_external_selected)[:, 1]

# Start with the COMPLETE original external dataset
all_results = external_df.copy()

# Add prediction and probability
all_results["Prediction"] = predictions
all_results["Probability"] = probabilities

# Sort by probability: highest = best predicted hit
all_results = all_results.sort_values(
    by="Probability",
    ascending=False
).reset_index(drop=True)

# Take TOP 20
top20_hits = all_results.head(20).copy()

# Display all columns
print("TOP 20 MTOR HITS")
print("Shape:", top20_hits.shape)

display(top20_hits)

TOP 20 MTOR HITS
Shape: (20, 3099)


,smiles,canonical_smiles,MolWt,LogP,H_Donors,H_Acceptors,TPSA,NumRotatableBonds,NumAtoms,HeavyAtomCount,...,atompair_1016,atompair_1017,atompair_1018,atompair_1019,atompair_1020,atompair_1021,atompair_1022,atompair_1023,Prediction,Probability
0,C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12,C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12,263.772,3.42750,2.0,3.0,50.94,5.0,18.0,18.0,...,1,1,1,0,1,1,1,0,1,0.737756
1,COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)...,COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)...,287.359,1.38290,2.0,4.0,50.72,1.0,21.0,21.0,...,1,1,0,0,0,0,0,0,1,0.663906
2,C[C@H]1O[C@@]2(CS1)CN1CCC2CC1,C[C@H]1O[C@@]2(CS1)CN1CCC2CC1,199.319,1.56020,0.0,3.0,12.47,0.0,13.0,13.0,...,0,0,0,0,0,0,0,0,1,0.655661
3,COc1cc(NC(C)CCCN)c2ncccc2c1,COc1cc(NC(C)CCCN)c2ncccc2c1,259.353,2.78270,2.0,4.0,60.17,6.0,19.0,19.0,...,1,1,1,1,1,1,0,0,1,0.654221
4,COc1cc(NC(C)CCCN)c2ncccc2c1,COc1cc(NC(C)CCCN)c2ncccc2c1,259.353,2.78270,2.0,4.0,60.17,6.0,19.0,19.0,...,1,1,1,1,1,1,0,0,1,0.654221
5,N#Cc1ccc2c(c1)CO[C@@]2(CCCN)c1ccc(F)cc1,N#Cc1ccc2c(c1)CO[C@@]2(CCCN)c1ccc(F)cc1,296.345,3.21008,1.0,3.0,59.04,4.0,22.0,22.0,...,1,1,1,0,1,1,0,0,1,0.646918
6,CCOc1ccc2nc3cc(N)ccc3c(N)c2c1,CCOc1ccc2nc3cc(N)ccc3c(N)c2c1,253.305,2.95110,2.0,4.0,74.16,2.0,19.0,19.0,...,1,1,1,1,1,1,0,0,1,0.640305
7,CCOc1ccc2nc3cc(N)ccc3c(N)c2c1,CCOc1ccc2nc3cc(N)ccc3c(N)c2c1,253.305,2.95110,2.0,4.0,74.16,2.0,19.0,19.0,...,1,1,1,1,1,1,0,0,1,0.640305
8,CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1,CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1,291.347,2.37310,2.0,5.0,71.70,7.0,21.0,21.0,...,1,1,1,1,1,0,0,0,1,0.634615
9,CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1,CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1,291.347,2.37310,2.0,5.0,71.70,7.0,21.0,21.0,...,1,1,1,1,1,0,0,0,1,0.634615


In [39]:
# Predict using ONLY the 1844 selected features
predictions = mtor_rf.predict(X_external_selected)
probabilities = mtor_rf.predict_proba(X_external_selected)[:, 1]

# Start with ONLY the 1844 selected features
results_selected = X_external_selected.copy()

# Add compound SMILES
results_selected.insert(
    0,
    "SMILES",
    external_df["smiles"].values
)

# Add prediction and probability
results_selected["Prediction"] = predictions
results_selected["Probability"] = probabilities

# Rank by probability
results_selected = results_selected.sort_values(
    by="Probability",
    ascending=False
).reset_index(drop=True)

# Keep Top 20
top20_selected = results_selected.head(20).copy()

# Add rank
top20_selected.insert(
    0,
    "Rank",
    range(1, 21)
)

print("Top 20 shape:", top20_selected.shape)

display(top20_selected)

Top 20 shape: (20, 1848)


,Rank,SMILES,MolWt,LogP,H_Donors,H_Acceptors,TPSA,NumRotatableBonds,NumHeteroatoms,NumNitrogens,...,morgan_2038,morgan_2039,morgan_2040,morgan_2042,morgan_2043,morgan_2044,morgan_2046,morgan_2047,Prediction,Probability
0,1,C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12,263.772,3.42750,2.0,3.0,50.94,5.0,4.0,3.0,...,0,0,0,0,0,0,0,0,1,0.737756
1,2,COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)...,287.359,1.38290,2.0,4.0,50.72,1.0,4.0,1.0,...,0,0,0,0,0,0,0,0,1,0.663906
2,3,C[C@H]1O[C@@]2(CS1)CN1CCC2CC1,199.319,1.56020,0.0,3.0,12.47,0.0,3.0,1.0,...,0,0,0,0,0,0,0,0,1,0.655661
3,4,COc1cc(NC(C)CCCN)c2ncccc2c1,259.353,2.78270,2.0,4.0,60.17,6.0,4.0,3.0,...,0,0,0,0,0,0,0,0,1,0.654221
4,5,COc1cc(NC(C)CCCN)c2ncccc2c1,259.353,2.78270,2.0,4.0,60.17,6.0,4.0,3.0,...,0,0,0,0,0,0,0,0,1,0.654221
5,6,N#Cc1ccc2c(c1)CO[C@@]2(CCCN)c1ccc(F)cc1,296.345,3.21008,1.0,3.0,59.04,4.0,4.0,2.0,...,0,0,0,0,0,0,0,0,1,0.646918
6,7,CCOc1ccc2nc3cc(N)ccc3c(N)c2c1,253.305,2.95110,2.0,4.0,74.16,2.0,4.0,3.0,...,0,0,0,0,0,0,0,0,1,0.640305
7,8,CCOc1ccc2nc3cc(N)ccc3c(N)c2c1,253.305,2.95110,2.0,4.0,74.16,2.0,4.0,3.0,...,0,0,0,0,0,0,0,0,1,0.640305
8,9,CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1,291.347,2.37310,2.0,5.0,71.70,7.0,5.0,1.0,...,1,0,0,0,0,0,0,0,1,0.634615
9,10,CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1,291.347,2.37310,2.0,5.0,71.70,7.0,5.0,1.0,...,1,0,0,0,0,0,0,0,1,0.634615


In [40]:
top20_selected.to_csv(
    "TOP20_MTOR_Hits_1844_Selected_Features.csv",
    index=False
)

print("✅ Top 20 MTOR hits saved successfully.")
print("File: TOP20_MTOR_Hits_1844_Selected_Features.csv")

✅ Top 20 MTOR hits saved successfully.
File: TOP20_MTOR_Hits_1844_Selected_Features.csv


In [41]:
from rdkit import Chem
import pandas as pd

# Start from your ranked results
unique_results = all_results.copy()

# Create canonical SMILES for reliable duplicate detection
unique_results["Canonical_SMILES"] = unique_results["smiles"].apply(
    lambda x: Chem.MolToSmiles(Chem.MolFromSmiles(x))
    if pd.notna(x) and Chem.MolFromSmiles(x) is not None
    else None
)

# Sort by probability so the highest-scoring duplicate is retained
unique_results = unique_results.sort_values(
    "Probability",
    ascending=False
)

# Remove exact duplicate chemical structures
unique_results = unique_results.drop_duplicates(
    subset="Canonical_SMILES",
    keep="first"
).reset_index(drop=True)

# Select 20 UNIQUE compounds
top20_unique = unique_results.head(20).copy()

# Reassign rank
top20_unique["Rank"] = range(1, len(top20_unique) + 1)

# Put Rank first
cols = ["Rank"] + [
    c for c in top20_unique.columns if c != "Rank"
]
top20_unique = top20_unique[cols]

print("Unique compounds available:", len(unique_results))
print("Final Top 20 unique hits:", len(top20_unique))

display(
    top20_unique[
        ["Rank", "smiles", "Prediction", "Probability"]
    ]
)

Unique compounds available: 56
Final Top 20 unique hits: 20


,Rank,smiles,Prediction,Probability
0,1,C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12,1,0.737756
1,2,COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)...,1,0.663906
2,3,C[C@H]1O[C@@]2(CS1)CN1CCC2CC1,1,0.655661
3,4,COc1cc(NC(C)CCCN)c2ncccc2c1,1,0.654221
4,5,N#Cc1ccc2c(c1)CO[C@@]2(CCCN)c1ccc(F)cc1,1,0.646918
5,6,CCOc1ccc2nc3cc(N)ccc3c(N)c2c1,1,0.640305
6,7,CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1,1,0.634615
7,8,CNCCCC(C#N)(c1ccc(O)c(OC)c1)C(C)C,1,0.619523
8,9,Cc1sc2ccccc2c1CC1=NCCN1,1,0.617215
9,10,COc1cc2c(cc1OC)C(=O)C(CC1CCNCC1)C2,1,0.605989


In [42]:
# Show ALL columns
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Remove the temporary Canonical_SMILES column from the final output
top20_full = top20_unique.drop(
    columns=["Canonical_SMILES"],
    errors="ignore"
).copy()

# Display ALL columns
print("Top 20 shape:", top20_full.shape)

display(top20_full)

Top 20 shape: (20, 3100)


Rank                                                     smiles  \
0      1                            C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12   
1      2  COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)NCC[C@@]341   
2      3                              C[C@H]1O[C@@]2(CS1)CN1CCC2CC1   
3      4                                COc1cc(NC(C)CCCN)c2ncccc2c1   
4      5                    N#Cc1ccc2c(c1)CO[C@@]2(CCCN)c1ccc(F)cc1   
5      6                              CCOc1ccc2nc3cc(N)ccc3c(N)c2c1   
6      7                         CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1   
7      8                          CNCCCC(C#N)(c1ccc(O)c(OC)c1)C(C)C   
8      9                                    Cc1sc2ccccc2c1CC1=NCCN1   
9     10                         COc1cc2c(cc1OC)C(=O)C(CC1CCNCC1)C2   
10    11                         CC(C)N(CCc1c[nH]c2ccc(O)cc12)C(C)C   
11    12                               CC[C@H](N)Cc1cc(OC)c(C)cc1OC   
12    13                              CC[C@@H](N)Cc1cc(OC)c(C)cc1OC   
13    14                                CCCCNCc1cc(=O)oc2cc(O)ccc12   
14    15                                 CCNCc1cc(=O)oc2cc(OC)ccc12   
15    16                            CC1(C)CC2C1C1CC1(C)C(O)CCC2(C)O   
16    17                               CCCCNCc1cc(=O)oc2cc(OC)ccc12   
17    18                      CNCCC[C@](C#N)(c1ccc(OC)c(OC)c1)C(C)C   
18    19                                       COc1cc2c(cc1OC)CNCC2   
19    20                                             CCC(O)C1CCCCN1   

                                             canonical_smiles    MolWt  \
0                             C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12  263.772   
1   COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)NCC[C@@]341  287.359   
2                               C[C@H]1O[C@@]2(CS1)CN1CCC2CC1  199.319   
3                                 COc1cc(NC(C)CCCN)c2ncccc2c1  259.353   
4                     N#Cc1ccc2c(c1)CO[C@@]2(CCCN)c1ccc(F)cc1  296.345   
5                               CCOc1ccc2nc3cc(N)ccc3c(N)c2c1  253.305   
6                          CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1  291.347   
7                           CNCCCC(C#N)(c1ccc(O)c(OC)c1)C(C)C  276.380   
8                                     Cc1sc2ccccc2c1CC1=NCCN1  230.336   
9                          COc1cc2c(cc1OC)C(=O)C(CC1CCNCC1)C2  289.375   
10                         CC(C)N(CCc1c[nH]c2ccc(O)cc12)C(C)C  260.381   
11                               CC[C@H](N)Cc1cc(OC)c(C)cc1OC  223.316   
12                              CC[C@@H](N)Cc1cc(OC)c(C)cc1OC  223.316   
13                                CCCCNCc1cc(=O)oc2cc(O)ccc12  247.294   
14                                 CCNCc1cc(=O)oc2cc(OC)ccc12  233.267   
15                            CC1(C)CC2C1C1CC1(C)C(O)CCC2(C)O  238.371   
16                               CCCCNCc1cc(=O)oc2cc(OC)ccc12  261.321   
17                      CNCCC[C@](C#N)(c1ccc(OC)c(OC)c1)C(C)C  290.407   
18                                       COc1cc2c(cc1OC)CNCC2  193.246   
19                                             CCC(O)C1CCCCN1  143.230   

       LogP  H_Donors  H_Acceptors   TPSA  NumRotatableBonds  NumAtoms  \
0   3.42750       2.0          3.0  50.94                5.0      18.0   
1   1.38290       2.0          4.0  50.72                1.0      21.0   
2   1.56020       0.0          3.0  12.47                0.0      13.0   
3   2.78270       2.0          4.0  60.17                6.0      19.0   
4   3.21008       1.0          3.0  59.04                4.0      22.0   
5   2.95110       2.0          4.0  74.16                2.0      19.0   
6   2.37310       2.0          5.0  71.70                7.0      21.0   
7   2.81778       2.0          4.0  65.28                7.0      20.0   
8   2.75392       1.0          3.0  24.39                2.0      16.0   
9   2.44850       1.0          4.0  47.56                4.0      21.0   
10  3.53480       2.0          2.0  39.26                5.0      19.0   
11  2.29202       1.0          3.0  44.48                5.0      16.

In [43]:
top20_full.to_csv(
    "TOP20_MTOR_Unique_Hits_ALL_Columns.csv",
    index=False
)

print("✅ Complete Top 20 unique MTOR hits saved.")
print("Rows:", len(top20_full))
print("Columns:", len(top20_full.columns))

✅ Complete Top 20 unique MTOR hits saved.
Rows: 20
Columns: 3100


In [44]:
# Extract Top 20 unique compounds with their SMILES

admet_input = top20_full[["Rank", "smiles"]].copy()

# Remove any missing SMILES
admet_input = admet_input.dropna(subset=["smiles"])

print("Compounds for ADMET:", len(admet_input))

display(admet_input)

Compounds for ADMET: 20


,Rank,smiles
0,1,C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12
1,2,COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)NCC[C@@]341
2,3,C[C@H]1O[C@@]2(CS1)CN1CCC2CC1
3,4,COc1cc(NC(C)CCCN)c2ncccc2c1
4,5,N#Cc1ccc2c(c1)CO[C@@]2(CCCN)c1ccc(F)cc1
5,6,CCOc1ccc2nc3cc(N)ccc3c(N)c2c1
6,7,CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1
7,8,CNCCCC(C#N)(c1ccc(O)c(OC)c1)C(C)C
8,9,Cc1sc2ccccc2c1CC1=NCCN1
9,10,COc1cc2c(cc1OC)C(=O)C(CC1CCNCC1)C2


In [45]:
admet_input.to_csv(
    "TOP20_MTOR_ADMET_Input.csv",
    index=False
)

print("✅ Saved:", len(admet_input), "compounds")
print("File: TOP20_MTOR_ADMET_Input.csv")

✅ Saved: 20 compounds
File: TOP20_MTOR_ADMET_Input.csv


In [46]:
# ============================================
# FINAL TOP 20 MTOR HITS
# Keep ALL original columns
# ============================================

# Remove temporary column used for duplicate checking
final_top20 = top20_unique.drop(
    columns=["Canonical_SMILES"],
    errors="ignore"
).copy()

# Make sure Rank is the first column
if "Rank" in final_top20.columns:
    final_top20 = final_top20[
        ["Rank"] +
        [c for c in final_top20.columns if c != "Rank"]
    ]

print("FINAL TOP 20 MTOR HITS")
print("Rows:", final_top20.shape[0])
print("Columns:", final_top20.shape[1])

display(final_top20)

FINAL TOP 20 MTOR HITS
Rows: 20
Columns: 3100


Rank                                                     smiles  \
0      1                            C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12   
1      2  COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)NCC[C@@]341   
2      3                              C[C@H]1O[C@@]2(CS1)CN1CCC2CC1   
3      4                                COc1cc(NC(C)CCCN)c2ncccc2c1   
4      5                    N#Cc1ccc2c(c1)CO[C@@]2(CCCN)c1ccc(F)cc1   
5      6                              CCOc1ccc2nc3cc(N)ccc3c(N)c2c1   
6      7                         CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1   
7      8                          CNCCCC(C#N)(c1ccc(O)c(OC)c1)C(C)C   
8      9                                    Cc1sc2ccccc2c1CC1=NCCN1   
9     10                         COc1cc2c(cc1OC)C(=O)C(CC1CCNCC1)C2   
10    11                         CC(C)N(CCc1c[nH]c2ccc(O)cc12)C(C)C   
11    12                               CC[C@H](N)Cc1cc(OC)c(C)cc1OC   
12    13                              CC[C@@H](N)Cc1cc(OC)c(C)cc1OC   
13    14                                CCCCNCc1cc(=O)oc2cc(O)ccc12   
14    15                                 CCNCc1cc(=O)oc2cc(OC)ccc12   
15    16                            CC1(C)CC2C1C1CC1(C)C(O)CCC2(C)O   
16    17                               CCCCNCc1cc(=O)oc2cc(OC)ccc12   
17    18                      CNCCC[C@](C#N)(c1ccc(OC)c(OC)c1)C(C)C   
18    19                                       COc1cc2c(cc1OC)CNCC2   
19    20                                             CCC(O)C1CCCCN1   

                                             canonical_smiles    MolWt  \
0                             C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12  263.772   
1   COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)NCC[C@@]341  287.359   
2                               C[C@H]1O[C@@]2(CS1)CN1CCC2CC1  199.319   
3                                 COc1cc(NC(C)CCCN)c2ncccc2c1  259.353   
4                     N#Cc1ccc2c(c1)CO[C@@]2(CCCN)c1ccc(F)cc1  296.345   
5                               CCOc1ccc2nc3cc(N)ccc3c(N)c2c1  253.305   
6                          CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1  291.347   
7                           CNCCCC(C#N)(c1ccc(O)c(OC)c1)C(C)C  276.380   
8                                     Cc1sc2ccccc2c1CC1=NCCN1  230.336   
9                          COc1cc2c(cc1OC)C(=O)C(CC1CCNCC1)C2  289.375   
10                         CC(C)N(CCc1c[nH]c2ccc(O)cc12)C(C)C  260.381   
11                               CC[C@H](N)Cc1cc(OC)c(C)cc1OC  223.316   
12                              CC[C@@H](N)Cc1cc(OC)c(C)cc1OC  223.316   
13                                CCCCNCc1cc(=O)oc2cc(O)ccc12  247.294   
14                                 CCNCc1cc(=O)oc2cc(OC)ccc12  233.267   
15                            CC1(C)CC2C1C1CC1(C)C(O)CCC2(C)O  238.371   
16                               CCCCNCc1cc(=O)oc2cc(OC)ccc12  261.321   
17                      CNCCC[C@](C#N)(c1ccc(OC)c(OC)c1)C(C)C  290.407   
18                                       COc1cc2c(cc1OC)CNCC2  193.246   
19                                             CCC(O)C1CCCCN1  143.230   

       LogP  H_Donors  H_Acceptors   TPSA  NumRotatableBonds  NumAtoms  \
0   3.42750       2.0          3.0  50.94                5.0      18.0   
1   1.38290       2.0          4.0  50.72                1.0      21.0   
2   1.56020       0.0          3.0  12.47                0.0      13.0   
3   2.78270       2.0          4.0  60.17                6.0      19.0   
4   3.21008       1.0          3.0  59.04                4.0      22.0   
5   2.95110       2.0          4.0  74.16                2.0      19.0   
6   2.37310       2.0          5.0  71.70                7.0      21.0   
7   2.81778       2.0          4.0  65.28                7.0      20.0   
8   2.75392       1.0          3.0  24.39                2.0      16.0   
9   2.44850       1.0          4.0  47.56                4.0      21.0   
10  3.53480       2.0          2.0  39.26                5.0      19.0   
11  2.29202       1.0          3.0  44.48                5.0      16.

In [47]:
final_top20.to_excel(
    "TOP20_MTOR_Hits_Final.xlsx",
    index=False
)

print("✅ FINAL TOP 20 MTOR HITS SAVED")
print("File: TOP20_MTOR_Hits_Final.xlsx")

ModuleNotFoundError: No module named 'openpyxl'

In [48]:
%pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy

In [49]:
final_top20.to_excel(
    "TOP20_MTOR_Hits_Final.xlsx",
    index=False,
    engine="openpyxl"
)

print("✅ FINAL TOP 20 MTOR HITS SAVED")
print("File: TOP20_MTOR_Hits_Final.xlsx")

✅ FINAL TOP 20 MTOR HITS SAVED
File: TOP20_MTOR_Hits_Final.xlsx


In [50]:
# Create clean ADMET input from final Top 20

admet_input = final_top20[["Rank", "smiles"]].copy()

admet_input = admet_input.dropna(subset=["smiles"])

print("Number of compounds for ADMET:", len(admet_input))

display(admet_input)

Number of compounds for ADMET: 20


,Rank,smiles
0,1,C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12
1,2,COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)NCC[C@@]341
2,3,C[C@H]1O[C@@]2(CS1)CN1CCC2CC1
3,4,COc1cc(NC(C)CCCN)c2ncccc2c1
4,5,N#Cc1ccc2c(c1)CO[C@@]2(CCCN)c1ccc(F)cc1
5,6,CCOc1ccc2nc3cc(N)ccc3c(N)c2c1
6,7,CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1
7,8,CNCCCC(C#N)(c1ccc(O)c(OC)c1)C(C)C
8,9,Cc1sc2ccccc2c1CC1=NCCN1
9,10,COc1cc2c(cc1OC)C(=O)C(CC1CCNCC1)C2


In [51]:
# Display the COMPLETE Top 20 MTOR hits
# including all original columns + Prediction + Probability

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

display(final_top20)

Rank                                                     smiles  \
0      1                            C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12   
1      2  COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)NCC[C@@]341   
2      3                              C[C@H]1O[C@@]2(CS1)CN1CCC2CC1   
3      4                                COc1cc(NC(C)CCCN)c2ncccc2c1   
4      5                    N#Cc1ccc2c(c1)CO[C@@]2(CCCN)c1ccc(F)cc1   
5      6                              CCOc1ccc2nc3cc(N)ccc3c(N)c2c1   
6      7                         CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1   
7      8                          CNCCCC(C#N)(c1ccc(O)c(OC)c1)C(C)C   
8      9                                    Cc1sc2ccccc2c1CC1=NCCN1   
9     10                         COc1cc2c(cc1OC)C(=O)C(CC1CCNCC1)C2   
10    11                         CC(C)N(CCc1c[nH]c2ccc(O)cc12)C(C)C   
11    12                               CC[C@H](N)Cc1cc(OC)c(C)cc1OC   
12    13                              CC[C@@H](N)Cc1cc(OC)c(C)cc1OC   
13    14                                CCCCNCc1cc(=O)oc2cc(O)ccc12   
14    15                                 CCNCc1cc(=O)oc2cc(OC)ccc12   
15    16                            CC1(C)CC2C1C1CC1(C)C(O)CCC2(C)O   
16    17                               CCCCNCc1cc(=O)oc2cc(OC)ccc12   
17    18                      CNCCC[C@](C#N)(c1ccc(OC)c(OC)c1)C(C)C   
18    19                                       COc1cc2c(cc1OC)CNCC2   
19    20                                             CCC(O)C1CCCCN1   

                                             canonical_smiles    MolWt  \
0                             C[C@H](CCCN)Nc1ccnc2cc(Cl)ccc12  263.772   
1   COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)NCC[C@@]341  287.359   
2                               C[C@H]1O[C@@]2(CS1)CN1CCC2CC1  199.319   
3                                 COc1cc(NC(C)CCCN)c2ncccc2c1  259.353   
4                     N#Cc1ccc2c(c1)CO[C@@]2(CCCN)c1ccc(F)cc1  296.345   
5                               CCOc1ccc2nc3cc(N)ccc3c(N)c2c1  253.305   
6                          CC(=O)c1cc2cccc(OCC(O)CNC(C)C)c2o1  291.347   
7                           CNCCCC(C#N)(c1ccc(O)c(OC)c1)C(C)C  276.380   
8                                     Cc1sc2ccccc2c1CC1=NCCN1  230.336   
9                          COc1cc2c(cc1OC)C(=O)C(CC1CCNCC1)C2  289.375   
10                         CC(C)N(CCc1c[nH]c2ccc(O)cc12)C(C)C  260.381   
11                               CC[C@H](N)Cc1cc(OC)c(C)cc1OC  223.316   
12                              CC[C@@H](N)Cc1cc(OC)c(C)cc1OC  223.316   
13                                CCCCNCc1cc(=O)oc2cc(O)ccc12  247.294   
14                                 CCNCc1cc(=O)oc2cc(OC)ccc12  233.267   
15                            CC1(C)CC2C1C1CC1(C)C(O)CCC2(C)O  238.371   
16                               CCCCNCc1cc(=O)oc2cc(OC)ccc12  261.321   
17                      CNCCC[C@](C#N)(c1ccc(OC)c(OC)c1)C(C)C  290.407   
18                                       COc1cc2c(cc1OC)CNCC2  193.246   
19                                             CCC(O)C1CCCCN1  143.230   

       LogP  H_Donors  H_Acceptors   TPSA  NumRotatableBonds  NumAtoms  \
0   3.42750       2.0          3.0  50.94                5.0      18.0   
1   1.38290       2.0          4.0  50.72                1.0      21.0   
2   1.56020       0.0          3.0  12.47                0.0      13.0   
3   2.78270       2.0          4.0  60.17                6.0      19.0   
4   3.21008       1.0          3.0  59.04                4.0      22.0   
5   2.95110       2.0          4.0  74.16                2.0      19.0   
6   2.37310       2.0          5.0  71.70                7.0      21.0   
7   2.81778       2.0          4.0  65.28                7.0      20.0   
8   2.75392       1.0          3.0  24.39                2.0      16.0   
9   2.44850       1.0          4.0  47.56                4.0      21.0   
10  3.53480       2.0          2.0  39.26                5.0      19.0   
11  2.29202       1.0          3.0  44.48                5.0      16.

In [52]:
print("Rows:", final_top20.shape[0])
print("Columns:", final_top20.shape[1])

print("\nLast columns:")
print(final_top20.columns[-5:].tolist())

Rows: 20
Columns: 3100

Last columns:
['atompair_1021', 'atompair_1022', 'atompair_1023', 'Prediction', 'Probability']


In [53]:
final_top20.to_excel(
    "TOP20_MTOR_Hits_Final.xlsx",
    index=False,
    engine="openpyxl"
)

print("✅ Complete Top 20 saved")

✅ Complete Top 20 saved


In [54]:
# Load the external dataset
external_df = pd.read_csv("atompair_addedBrain.csv")

# Load the 561 GSK3B-selected features
selected_df = pd.read_csv("rfecv_selected_featuresGSK3B.csv")

selected_features = selected_df["feature"].tolist()

print("Selected GSK3B features:", len(selected_features))

# Select ONLY the 561 features
X_external_GSK3B = external_df[selected_features]

print("External selected matrix:", X_external_GSK3B.shape)

Selected GSK3B features: 561
External selected matrix: (77, 561)
